# Robinson Crusoe Dataset Preparation

This notebook prepares a balanced dataset for training a neural network to identify Robinson Crusoe adaptations.

## Dataset Components:
- **Positive class (1)**: Robinson Crusoe adaptations from HathiTrust (1,484 texts)
- **Negative class (0)**: Random 18th-century texts from ECCO-TCP (sampled to match)

## Outputs:
- Balanced HDF5 dataset split into train/validation/test sets
- Exploratory data analysis and statistics

In [ ]:
import numpy as np
from pathlib import Path
import re
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm import tqdm
import warnings
warnings.filterwarnings('ignore')

# Set random seed for reproducibility
np.random.seed(42)

# Visualization settings
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)

print("Environment setup complete")
print(f"NumPy version: {np.__version__}")
print(f"Pandas version: {pd.__version__}")

## 1. Load Robinson Crusoe Adaptations

In [ ]:
def clean_text(text):
    """Clean text by removing non-alphanumeric characters except basic punctuation."""
    # Remove non-alphanumeric characters (keep spaces, periods, quotes, forward slashes)
    text = re.sub(r'[^a-zA-Z0-9. /"]', r'', text)
    
    # Remove excessive whitespace
    text = re.sub(r'\s+', ' ', text)
    
    return text.strip()

# Check if Adaptations.zip exists and extract if needed
if not Path('./Adaptations').exists():
    print("Adaptations directory not found. Checking for zip file...")
    if Path('./Adaptations.zip').exists():
        import zipfile
        print("Extracting Adaptations.zip...")
        with zipfile.ZipFile('./Adaptations.zip', 'r') as zip_ref:
            zip_ref.extractall('.')
        print("Extraction complete!")
    else:
        print("WARNING: Adaptations.zip not found. Please ensure the data is available.")

# Load Robinson Crusoe texts
print("\nLoading Robinson Crusoe adaptations...")
RC_docs = []
RC_files = []

for file in tqdm(list(Path("./Adaptations").rglob("*.txt")), desc="Loading RC texts"):
    try:
        with open(file, encoding="ISO-8859-1") as f:
            txt_file_as_string = f.read()
            txt_file_as_string = clean_text(txt_file_as_string)
            
            RC_docs.append([txt_file_as_string, 1])  # Label 1 for adaptations
            RC_files.append(file.name)
    except Exception as e:
        print(f"Error reading {file.name}: {e}")
        continue

print(f"\nLoaded {len(RC_docs)} Robinson Crusoe adaptation texts")

In [ ]:
# Create dataframe for RC texts
df_rc = pd.DataFrame(RC_docs, columns=['text', 'label'])
df_rc['filename'] = RC_files
df_rc['text_length'] = df_rc['text'].apply(len)
df_rc['word_count'] = df_rc['text'].apply(lambda x: len(x.split()))

print("\nRobinson Crusoe Dataset Statistics:")
print("=" * 70)
print(f"Total texts: {len(df_rc)}")
print(f"\nText length (characters):")
print(df_rc['text_length'].describe())
print(f"\nWord count:")
print(df_rc['word_count'].describe())

# Display sample
print("\nSample entries:")
print(df_rc[['filename', 'text_length', 'word_count', 'label']].head())

## 2. Load Random Texts (ECCO-TCP)

In [ ]:
# Check if Random.zip exists and extract if needed
if not Path('./Random').exists():
    print("Random directory not found. Checking for zip file...")
    if Path('./Random.zip').exists():
        import zipfile
        print("Extracting Random.zip...")
        with zipfile.ZipFile('./Random.zip', 'r') as zip_ref:
            zip_ref.extractall('.')
        print("Extraction complete!")
    else:
        print("WARNING: Random.zip not found. Please ensure the data is available.")

# Load random texts
print("\nLoading random 18th-century texts...")
random_docs = []
random_files = []

for file in tqdm(list(Path("./Random").rglob("*.txt")), desc="Loading random texts"):
    try:
        with open(file, encoding="ISO-8859-1") as f:
            txt_file_as_string = f.read()
            txt_file_as_string = clean_text(txt_file_as_string)
            
            random_docs.append([txt_file_as_string, 0])  # Label 0 for non-adaptations
            random_files.append(file.name)
    except Exception as e:
        print(f"Error reading {file.name}: {e}")
        continue

print(f"\nLoaded {len(random_docs)} random texts")

In [ ]:
# Create dataframe for random texts
df_random_full = pd.DataFrame(random_docs, columns=['text', 'label'])
df_random_full['filename'] = random_files
df_random_full['text_length'] = df_random_full['text'].apply(len)
df_random_full['word_count'] = df_random_full['text'].apply(lambda x: len(x.split()))

print("\nRandom Texts Dataset Statistics (before sampling):")
print("=" * 70)
print(f"Total texts: {len(df_random_full)}")
print(f"\nText length (characters):")
print(df_random_full['text_length'].describe())
print(f"\nWord count:")
print(df_random_full['word_count'].describe())

## 3. Balance the Dataset

We'll sample the random texts to match the number of Robinson Crusoe adaptations, creating a balanced dataset.

In [ ]:
# Sample random texts to match RC count
n_rc_texts = len(df_rc)
print(f"Sampling {n_rc_texts} random texts to match Robinson Crusoe count...")

df_random = df_random_full.sample(n=n_rc_texts, random_state=42)

print(f"\nSampled {len(df_random)} random texts")
print("\nRandom Texts Dataset Statistics (after sampling):")
print("=" * 70)
print(f"\nText length (characters):")
print(df_random['text_length'].describe())
print(f"\nWord count:")
print(df_random['word_count'].describe())

In [ ]:
# Combine datasets
df = pd.concat([df_random, df_rc], ignore_index=True)

# Shuffle the combined dataset
df = df.sample(frac=1, random_state=42).reset_index(drop=True)

print("\nCombined Dataset:")
print("=" * 70)
print(f"Total samples: {len(df)}")
print(f"\nClass distribution:")
print(df['label'].value_counts().sort_index())
print(f"\nClass balance: {df['label'].value_counts(normalize=True).round(3).to_dict()}")

# Display sample
print("\nSample of combined dataset:")
print(df[['filename', 'text_length', 'word_count', 'label']].head(10))

## 4. Exploratory Data Analysis

In [ ]:
# Visualize text length distributions
fig, axes = plt.subplots(2, 2, figsize=(16, 12))

# Text length distribution by class
df[df['label'] == 0]['text_length'].hist(bins=50, alpha=0.7, label='Random', 
                                           color='blue', ax=axes[0, 0])
df[df['label'] == 1]['text_length'].hist(bins=50, alpha=0.7, label='RC Adaptation', 
                                           color='orange', ax=axes[0, 0])
axes[0, 0].set_xlabel('Text Length (characters)', fontsize=12)
axes[0, 0].set_ylabel('Frequency', fontsize=12)
axes[0, 0].set_title('Text Length Distribution by Class', fontsize=14, fontweight='bold')
axes[0, 0].legend()
axes[0, 0].grid(alpha=0.3)

# Word count distribution by class
df[df['label'] == 0]['word_count'].hist(bins=50, alpha=0.7, label='Random', 
                                         color='blue', ax=axes[0, 1])
df[df['label'] == 1]['word_count'].hist(bins=50, alpha=0.7, label='RC Adaptation', 
                                         color='orange', ax=axes[0, 1])
axes[0, 1].set_xlabel('Word Count', fontsize=12)
axes[0, 1].set_ylabel('Frequency', fontsize=12)
axes[0, 1].set_title('Word Count Distribution by Class', fontsize=14, fontweight='bold')
axes[0, 1].legend()
axes[0, 1].grid(alpha=0.3)

# Box plot for text length
df.boxplot(column='text_length', by='label', ax=axes[1, 0])
axes[1, 0].set_xlabel('Class (0=Random, 1=RC Adaptation)', fontsize=12)
axes[1, 0].set_ylabel('Text Length (characters)', fontsize=12)
axes[1, 0].set_title('Text Length Comparison', fontsize=14, fontweight='bold')
axes[1, 0].get_figure().suptitle('')  # Remove default title

# Box plot for word count
df.boxplot(column='word_count', by='label', ax=axes[1, 1])
axes[1, 1].set_xlabel('Class (0=Random, 1=RC Adaptation)', fontsize=12)
axes[1, 1].set_ylabel('Word Count', fontsize=12)
axes[1, 1].set_title('Word Count Comparison', fontsize=14, fontweight='bold')
axes[1, 1].get_figure().suptitle('')  # Remove default title

plt.tight_layout()
plt.savefig('dataset_distribution_analysis.png', dpi=300, bbox_inches='tight')
plt.show()

print("Visualization saved as 'dataset_distribution_analysis.png'")

In [ ]:
# Statistical comparison between classes
print("\nStatistical Comparison Between Classes:")
print("=" * 70)

for metric in ['text_length', 'word_count']:
    print(f"\n{metric.upper()}:")
    print("-" * 70)
    
    random_stats = df[df['label'] == 0][metric].describe()
    rc_stats = df[df['label'] == 1][metric].describe()
    
    comparison_df = pd.DataFrame({
        'Random Texts': random_stats,
        'RC Adaptations': rc_stats,
        'Difference': rc_stats - random_stats
    })
    
    print(comparison_df)
    
    # Perform t-test
    from scipy import stats
    t_stat, p_value = stats.ttest_ind(
        df[df['label'] == 0][metric],
        df[df['label'] == 1][metric]
    )
    print(f"\nt-test: t-statistic={t_stat:.4f}, p-value={p_value:.4e}")
    if p_value < 0.05:
        print(f"✓ Significant difference between classes (p < 0.05)")
    else:
        print(f"✗ No significant difference between classes (p >= 0.05)")

## 5. Data Quality Checks

In [ ]:
print("\nData Quality Checks:")
print("=" * 70)

# Check for missing values
print("\nMissing values:")
print(df.isnull().sum())

# Check for empty texts
empty_texts = df[df['text_length'] == 0]
print(f"\nEmpty texts: {len(empty_texts)}")
if len(empty_texts) > 0:
    print("WARNING: Found empty texts! Removing...")
    df = df[df['text_length'] > 0].reset_index(drop=True)
    print(f"Remaining samples: {len(df)}")

# Check for very short texts (potential data quality issues)
min_length = 100  # characters
short_texts = df[df['text_length'] < min_length]
print(f"\nTexts with < {min_length} characters: {len(short_texts)}")
if len(short_texts) > 0:
    print(f"  Class 0 (Random): {len(short_texts[short_texts['label'] == 0])}")
    print(f"  Class 1 (RC Adaptation): {len(short_texts[short_texts['label'] == 1])}")
    print("\nConsider removing very short texts if they represent data quality issues.")

# Check for duplicates
duplicates = df.duplicated(subset=['text'], keep=False)
print(f"\nDuplicate texts: {duplicates.sum()}")
if duplicates.sum() > 0:
    print("WARNING: Found duplicate texts! Removing...")
    df = df.drop_duplicates(subset=['text'], keep='first').reset_index(drop=True)
    print(f"Remaining samples: {len(df)}")
    print(f"Updated class distribution:")
    print(df['label'].value_counts().sort_index())

print("\n✓ Data quality checks complete")

## 6. Save Dataset to HDF5

We'll save the cleaned and balanced dataset for use in model training.

In [ ]:
# Keep only necessary columns for training
df_final = df[['text', 'label']].copy()

print("\nFinal Dataset Summary:")
print("=" * 70)
print(f"Total samples: {len(df_final)}")
print(f"\nClass distribution:")
print(df_final['label'].value_counts().sort_index())
print(f"\nClass balance:")
for label, count in df_final['label'].value_counts(normalize=True).sort_index().items():
    print(f"  Class {label}: {count:.1%}")

# Save to HDF5
output_file = 'training_set.h5'
print(f"\nSaving dataset to {output_file}...")

df_final.to_hdf(output_file, key='balanced', mode='w', format='table')

print(f"✓ Dataset saved successfully!")
print(f"\nFile: {output_file}")
print(f"Size: {Path(output_file).stat().st_size / (1024*1024):.2f} MB")

In [ ]:
# Verify saved dataset
print("\nVerifying saved dataset...")
df_loaded = pd.read_hdf(output_file, 'balanced')

print(f"\nLoaded dataset shape: {df_loaded.shape}")
print(f"Columns: {df_loaded.columns.tolist()}")
print(f"\nClass distribution:")
print(df_loaded['label'].value_counts().sort_index())

# Check if loaded data matches original
if df_loaded.equals(df_final):
    print("\n✓ Verification successful! Loaded data matches original.")
else:
    print("\n⚠ WARNING: Loaded data differs from original!")

## 7. Dataset Summary Report

In [ ]:
# Generate summary report
summary_report = f"""
{'='*80}
ROBINSON CRUSOE ADAPTATION DETECTION - DATASET SUMMARY
{'='*80}

Dataset: {output_file}
Created: {pd.Timestamp.now().strftime('%Y-%m-%d %H:%M:%S')}

DATASET COMPOSITION:
{'-'*80}
Total samples: {len(df_final):,}

Class 0 (Random 18th-century texts): {(df_final['label'] == 0).sum():,} ({(df_final['label'] == 0).sum() / len(df_final):.1%})
Class 1 (RC Adaptations): {(df_final['label'] == 1).sum():,} ({(df_final['label'] == 1).sum() / len(df_final):.1%})

TEXT STATISTICS:
{'-'*80}
Average text length: {df['text_length'].mean():,.0f} characters
Average word count: {df['word_count'].mean():,.0f} words

Min text length: {df['text_length'].min():,} characters
Max text length: {df['text_length'].max():,} characters

RECOMMENDED SPLITS:
{'-'*80}
Training set (70%): {int(len(df_final) * 0.7):,} samples
Validation set (15%): {int(len(df_final) * 0.15):,} samples
Test set (15%): {int(len(df_final) * 0.15):,} samples

DATA SOURCES:
{'-'*80}
- Robinson Crusoe Adaptations: HathiTrust Digital Library
- Random Texts: ECCO-TCP (Eighteenth Century Collections Online)

PREPROCESSING APPLIED:
{'-'*80}
- Removed non-alphanumeric characters (kept spaces, periods, quotes, slashes)
- Removed excessive whitespace
- Removed empty texts
- Removed duplicates
- Balanced classes by sampling

{'='*80}
"""

print(summary_report)

# Save report to file
with open('dataset_summary.txt', 'w') as f:
    f.write(summary_report)

print("\n✓ Summary report saved to 'dataset_summary.txt'")

## Next Steps

The balanced dataset is now ready for model training. Proceed to `train.ipynb` to:
1. Split the data into train/validation/test sets
2. Generate embeddings using Universal Sentence Encoder
3. Train a neural network classifier
4. Evaluate model performance